# HW3 — сохранить результаты в Git

Запустите эту ячейку после `HW3_pipeline.ipynb` и `HW3_metrics.ipynb`. Она копирует готовые графы и метрики из игнорируемых папок в `hw3/results/ganoshenko/`. Каждый новый запуск после предобработки получает свою папку по идентификатору запуска. Большие промежуточные файлы GraphRAG не копируются.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil

root = Path.cwd().resolve()
if root.name == 'hw3':
    root = root.parent
run_pointer = root / 'hw3_runs/ganoshenko/project_path.txt'
if not run_pointer.is_file():
    raise FileNotFoundError(f'Сначала выполните HW3_pipeline.ipynb: {run_pointer}')
after = Path(run_pointer.read_text(encoding='utf-8').strip()).resolve()
run_info = json.loads((after / 'run_info.json').read_text(encoding='utf-8'))
if not (after / 'hw3_complete.json').is_file():
    raise RuntimeError(f'Запуск ещё не завершён: {after}')
before = Path(run_info['baseline']).resolve()
runs_root = (root / 'local_runs').resolve()
if before.parent != runs_root or after.parent != runs_root:
    raise ValueError('Пути запусков должны находиться непосредственно в local_runs/')
results = root / 'hw3/results/ganoshenko'
baseline_dir = results / 'baseline'
snapshot_dir = results / 'runs' / after.name
metrics_source = root / 'hw3_runs/ganoshenko'
for relative in ('comparison/comparison.json', 'comparison/retrieval.json', 'comparison/llm_judge.json'):
    data = json.loads((metrics_source / relative).read_text(encoding='utf-8'))
    if Path(data['after']['run']).name != after.name:
        raise RuntimeError(f'Метрики {relative} относятся к другому запуску')

def portable(value):
    if isinstance(value, dict):
        return {key: portable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [portable(item) for item in value]
    if isinstance(value, str) and value.lower().startswith(str(root).lower()):
        return value[len(str(root)):].lstrip('\\/').replace('\\', '/')
    return value

def archive(source, destination):
    if not source.is_file():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.suffix == '.json':
        data = portable(json.loads(source.read_text(encoding='utf-8')))
        destination.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    else:
        shutil.copy2(source, destination)
    return destination

exported = []
for run, destination in ((before, baseline_dir), (after, snapshot_dir)):
    for relative in ('graph.html', 'output/normalized/graph.graphml'):
        exported.append(archive(run / relative, destination / Path(relative).name))
for relative in (
    'report.json',
    'stage_metrics/stage_metrics.md',
    'stage_metrics/stage_metrics.json',
    'stage_metrics/technical_source_audit.csv',
    'comparison/comparison.md',
    'comparison/comparison.json',
    'comparison/retrieval.json',
    'comparison/llm_judge.json',
):
    exported.append(archive(metrics_source / relative, snapshot_dir / relative))
manifest = {
    'baseline_run': before.name,
    'processed_run': after.name,
    'baseline_input_sha256': run_info['baseline_input_sha256'],
    'prepared_chunks_sha256': run_info['prepared_chunks_sha256'],
    'completion_model': run_info['completion_model'],
    'embedding_model': run_info['embedding_model'],
    'files': {
        str(path.relative_to(results)).replace('\\', '/'): {
            'bytes': path.stat().st_size,
            'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
        }
        for path in exported
    },
}
(snapshot_dir / 'manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print(f'Сохранено {len(exported)} файлов: {snapshot_dir.relative_to(root)}')
